# HeartTwin synthetic BSP and bounded mechanics verification
Run in Edge Colab. Every mesh, signal and observation in these tests is manufactured; no patient data are downloaded or required.
Pins reproduce the tested implementation bodies. HeartTwin's candidate mechanics override is installed explicitly for this experiment; the merged stack lock advances separately to the same tested body with evidence/docs.
Observed constituent runs: 195 HeartTwin patient tests and 100 CardiMech tests passed, no skips. Traced assembly allocation fell ~85% on one 3,078-node/12,288-tet synthetic shell. No clinical accuracy, full KIT capacity, or total-process-memory claim.
The consolidated notebook is compiled separately; it is not claimed to have been replayed end to end after assembly. Existing failed setup/lint logs remain in the archived evidence.


In [ ]:
from pathlib import Path
import os, subprocess, sys, time
ENG=Path("/content")/("HeartTwin_synthetic_completion_"+time.strftime("%Y%m%d_%H%M%S"))
ENG.mkdir()
subprocess.run([sys.executable,"-m","pip","install","virtualenv"],check=True)
envdir=ENG/"venv"
subprocess.run([sys.executable,"-m","virtualenv","--system-site-packages",str(envdir)],check=True)
engpython=envdir/"bin/python"
engenv=dict(os.environ,PATH=str(envdir/"bin")+os.pathsep+os.environ.get("PATH",""),OPENBLAS_NUM_THREADS="1",MPLBACKEND="Agg")
ht,mech=ENG/"Virelion-HeartTwin",ENG/"Virelion-CardiMech"
for directory,revision in [(ht,"4ac8b01c30c6d746254ab7cea75019e6fee42bf4"),(mech,"47753cb5141f07c27b6cc64d64aaab11f7630e48")]:
    subprocess.run(["git","clone","https://github.com/Virelion-Biotech/"+directory.name+".git",str(directory)],check=True)
    subprocess.run(["git","-C",str(directory),"checkout",revision],check=True)
subprocess.run([str(engpython),"-m","pip","install","--ignore-installed","--no-deps","ruff"],check=True)
wanted=["CardiAnatomy","CardiEP","CardiInfer","CardiFlow","ElectroTrace"]
component_urls=[s for s in (ht/"requirements-services.txt").read_text().splitlines() if s.startswith("git+") and any("/Virelion-"+name+".git@" in s for name in wanted)]
with (ENG/"install.log").open("w") as log:
    result=subprocess.run([str(engpython),"-m","pip","install","numpy==2.1.3","scipy==1.16.3","-e",str(ht)+"[dev,patient-native]","-e",str(mech)+"[dev,spatial,validation]",*component_urls],env=engenv,stdout=log,stderr=subprocess.STDOUT)
print((ENG/"install.log").read_text()[-1500:])
result.check_returncode()



In [ ]:
import json,subprocess
results={}
commands={
"patient":(ht,[str(engpython),"-m","pytest","-q",*[str(p.relative_to(ht)) for p in sorted((ht/"tests").glob("test_patient*.py"))],"--junitxml="+str(ENG/"patient-final.xml")]),
"mechanics":(mech,[str(engpython),"-m","pytest","-q","--junitxml="+str(ENG/"mech-final.xml")]),
"hearttwin_lint":(ht,[str(engpython),"-m","ruff","check","hearttwin/patient_volume_conductor.py","hearttwin/patient_spatial_ep.py","hearttwin/patient_bsp.py","hearttwin/patient_preparation.py","tests/test_patient_bsp.py"]),
"mechanics_lint":(mech,[str(engpython),"-m","ruff","check","src/cardimech/fem_backend.py","src/cardimech/fem_materials.py","tests/test_fem_bounded.py"])}
for name,(directory,command) in commands.items():
    with (ENG/(name+".log")).open("w") as log:
        p=subprocess.run(command,cwd=directory,env=engenv,stdout=log,stderr=subprocess.STDOUT)
    results[name]=p.returncode
    print(name,p.returncode,(ENG/(name+".log")).read_text()[-1500:])
(ENG/"run_status.json").write_text(json.dumps(results,indent=2))
assert not any(results.values())



In [ ]:
import urllib.request,subprocess
(ENG/"profile_mechanics.py").write_text("\"\"\"Synthetic bounded-assembly stress comparison; no patient data or guard bypass.\"\"\"\nimport importlib.util\nimport json\nimport time\nimport tracemalloc\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom cardimech.artifacts import write_json_artifact\nfrom cardimech.fem_backend import assemble\nfrom cardimech.fem_examples import shell_mesh\nfrom cardimech.fem_mesh import load_mesh\n\nroot = Path(__file__).parent\nspec = importlib.util.spec_from_file_location(\"cardimech.legacy_profile\", root/\"legacy_fem_backend.py\")\nlegacy = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(legacy)\npayload = shell_mesh(4, layers=3)\nref = write_json_artifact(root, filename=\"synthetic-stress-mesh.json\", artifact_id=\"synthetic-stress\",\n                          kind=\"tetra_mechanics_mesh\", payload=payload)\nmesh = load_mesh(ref, payload[\"subject_id\"])\nu = np.random.default_rng(78).normal(0, 1e-7, mesh.nodes.shape)\nreport = {\"synthetic_only\": True, \"nodes\": len(mesh.nodes), \"tetrahedra\": len(mesh.elements),\n          \"within_unchanged_mesh_limits\": len(mesh.nodes)<=5000 and len(mesh.elements)<=20000,\n          \"measurement\": \"tracemalloc peak allocation during assembly, not process RSS\",\n          \"baseline_commit\": \"5dc14d886ccbda5a1398b863f9ae7821bbffa18b\", \"materials\": {}}\nfor model, parameters in [\n        (\"neo_hookean\", {\"mu\": 1000, \"kappa\": 2000}),\n        (\"guccione\", {\"c\": 400, \"bf\": 8, \"bt\": 2, \"bfs\": 3, \"kappa\": 2000})]:\n    results, profiles = [], {}\n    for name, function in [(\"legacy\", legacy.assemble), (\"bounded\", assemble)]:\n        tracemalloc.start()\n        start = time.perf_counter()\n        result = function(mesh, u, model, parameters, 20, 2)\n        elapsed = time.perf_counter()-start\n        current, peak = tracemalloc.get_traced_memory()\n        tracemalloc.stop()\n        results.append(result)\n        profiles[name] = {\"seconds\": elapsed, \"peak_traced_bytes\": peak}\n    old, new = results\n    delta = old[1]-new[1]\n    max_k = float(np.max(abs(delta.data), initial=0))\n    max_r = float(np.max(abs(old[0]-new[0])))\n    max_fields = [float(np.max(abs(a-b))) for a,b in zip(old[2],new[2],strict=True)]\n    assert max_k<1e-9 and max_r<1e-11 and max(max_fields)<1e-10\n    profiles[\"maximum_tangent_difference\"] = max_k\n    profiles[\"maximum_residual_difference_N\"] = max_r\n    profiles[\"maximum_field_differences\"] = max_fields\n    profiles[\"peak_traced_allocation_ratio\"] = profiles[\"bounded\"][\"peak_traced_bytes\"]/profiles[\"legacy\"][\"peak_traced_bytes\"]\n    report[\"materials\"][model] = profiles\n(root/\"mechanics_stress.json\").write_text(json.dumps(report, indent=2, allow_nan=False))\nprint(json.dumps(report, indent=2))\n")
(ENG/"legacy_fem_backend.py").write_bytes(urllib.request.urlopen("https://raw.githubusercontent.com/Virelion-Biotech/Virelion-CardiMech/5dc14d886ccbda5a1398b863f9ae7821bbffa18b/src/cardimech/fem_backend.py").read())
p=subprocess.run([str(engpython),str(ENG/"profile_mechanics.py")],cwd=mech,env=engenv,text=True,capture_output=True)
(ENG/"mechanics-stress.log").write_text(p.stdout+"\n"+p.stderr)
print(p.stdout)
p.check_returncode()



In [ ]:
import zipfile,hashlib,json
archive=ENG/"synthetic_completion_evidence.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in ENG.iterdir():
        if p.is_file() and p.suffix in [".json",".log",".xml",".py"]:
            z.write(p,p.name)
print("Archive",archive,"SHA256",hashlib.sha256(archive.read_bytes()).hexdigest())
from google.colab import files
files.download(str(archive))

